LAYER 1

In [1]:
# 0. Imports
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import timm
import numpy as np
import pandas as pd
from PIL import Image
from timm.data import create_transform
# Ensure scripts folder is in path
import scripts.model
from scripts.prepare_data import prepare_data
from scripts.datasets import GeoguessrDataset
from scripts.model import GeoguessrModel
import importlib
from scripts.losses import CoordinateLoss, haversine_distance, latlon_to_cartesian

In [2]:
# 1. SETUP AND DATA PREPARATION
print("Checking data preparation...")
prepare_data() # <--- This now perfectly clusters your data automatically!
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
print(f"Using device: {device}")
os.makedirs("saved_models", exist_ok=True)
base_dir = os.path.abspath('.')
csv_path = os.path.join(base_dir, 'training_dataset', 'noised_dataset', 'clustered_training_data.csv')
image_dir = os.path.join(base_dir, 'training_dataset', 'noised_dataset', 'images')
print("\nLoading dataset...")
dataset = GeoguessrDataset(
    csv_path=csv_path, image_dir=image_dir, transform=None
)
num_countries = dataset.get_num_classes() # This will dynamically pull 160 clusters!
print(f"\nDataset loaded successfully with {len(dataset)} images and {num_countries} geographic clusters.")

Checking data preparation...
--> Found c:\Users\Yash T\Desktop\geoguessr\geolocation-prediction\training_dataset\noised_dataset\clustered_training_data.csv. Skipping K-Means generation.
Using device: cuda

Loading dataset...

Dataset loaded successfully with 19002 images and 160 geographic clusters.


In [3]:
# 2. EVALUATION FUNCTION
def evaluate_model(model, dataset, batch_size=64, workers=8):
    print("\n--- Evaluating Layer 1 (Cluster Classifier) ---")
    model.eval()
    eval_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=workers)
    criterion = nn.CrossEntropyLoss()
    
    total_loss = 0.0
    correct = 0
    total = 0
    progress_bar = tqdm(eval_loader, desc="Evaluating")
    
    with torch.no_grad():
        for batch in progress_bar:
            images = batch['image'].to(device, non_blocking=True)
            labels = batch['country_label'].to(device, non_blocking=True)
            
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                country_logits = outputs['country_logits']
                loss = criterion(country_logits, labels)
            
            total_loss += loss.item()
            predictions = torch.argmax(country_logits, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
            
    avg_loss = total_loss / len(eval_loader)
    accuracy = (correct / total) * 100
    print(f"--> Evaluation Complete! Avg Loss: {avg_loss:.4f} | Accuracy: {accuracy:.2f}%\n")

In [4]:
# 3. THE EXPERIMENT COMMAND CENTER (LAYER 1)
def run_experiment(backbone_name, batch_size=64, epochs=3, workers=8, learning_rate=1e-4):
    print(f"\n{'='*50}")
    print(f"STARTING LAYER 1 TRAINING: {backbone_name}")
    print(f"{'='*50}")
    
    dummy_model = timm.create_model(backbone_name, pretrained=False)
    data_config = timm.data.resolve_data_config({}, model=dummy_model)
    required_image_size = data_config['input_size'][1] 
    
    transform = A.Compose([
        A.Resize(required_image_size, required_image_size),
        A.Normalize(mean=data_config['mean'], std=data_config['std']),
        ToTensorV2()
    ])
    dataset.transform = transform
    
    model = GeoguessrModel(num_countries=num_countries, backbone_name=backbone_name, pretrained=True).to(device)
    
    # MODULAR RESUME
    save_path = f"saved_models/layer1_{backbone_name}.pth"
    if os.path.exists(save_path):
        print(f"--> Found existing Layer 1 weights! Loading {save_path} to resume.")
        layer1_weights = torch.load(save_path, map_location=device, weights_only=True)
        model.backbone.load_state_dict(layer1_weights['backbone'], strict=False)
        model.country_head.load_state_dict(layer1_weights['country_head'], strict=False)
    else:
        print("--> No existing Layer 1 weights found. Starting fresh.")
        
    if epochs == 0:
        evaluate_model(model, dataset, batch_size)
        return model
        
    persist = True if workers > 0 else False
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=workers, persistent_workers=persist)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
    scaler = torch.amp.GradScaler('cuda')
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs} [{backbone_name}]")
        
        for batch in progress_bar:
            images = batch['image'].to(device, non_blocking=True)
            labels = batch['country_label'].to(device, non_blocking=True)
            
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs['country_logits'], labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item()
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        epoch_loss = running_loss / len(dataloader)
        print(f"--> Epoch {epoch+1} Average Training Loss: {epoch_loss:.4f}")
        
        # SAVE STRICTLY LAYER 1
        torch.save({'backbone': model.backbone.state_dict(), 'country_head': model.country_head.state_dict()}, save_path)
        
    print(f"SUCCESS! Layer 1 safely saved to: {save_path}")
    evaluate_model(model, dataset, batch_size)
    return model

In [8]:
# 4. RUN YOUR EXPERIMENTS HERE
model_v2_small = run_experiment(backbone_name='efficientnetv2_rw_s', batch_size=40, epochs=5, learning_rate=7e-5)


STARTING LAYER 1 TRAINING: efficientnetv2_rw_s
--> No existing Layer 1 weights found. Starting fresh.


Epoch 1/5 [efficientnetv2_rw_s]: 100%|██████████| 476/476 [02:06<00:00,  3.77it/s, loss=5.1777]


--> Epoch 1 Average Training Loss: 4.7028


Epoch 2/5 [efficientnetv2_rw_s]: 100%|██████████| 476/476 [01:51<00:00,  4.26it/s, loss=5.0898]


--> Epoch 2 Average Training Loss: 3.7793


...

--> Epoch 4 Average Training Loss: 2.2124


Epoch 5/5 [efficientnetv2_rw_s]: 100%|██████████| 476/476 [01:56<00:00,  4.07it/s, loss=3.7832]


--> Epoch 5 Average Training Loss: 1.4752
SUCCESS! Layer 1 safely saved to: saved_models/layer1_efficientnetv2_rw_s.pth

--- Evaluating Layer 1 (Cluster Classifier) ---


Evaluating: 100%|██████████| 476/476 [01:06<00:00,  7.17it/s]


--> Evaluation Complete! Avg Loss: 0.8813 | Accuracy: 84.78%



LAYER 2

In [9]:
# 5. LAYER 2 TRAINING
def train_phase2_coordinates(model, dataset, backbone_name, epochs=5, batch_size=64, workers=8, lr=1e-4):
    print("\n" + "="*50)
    print(f"STARTING PHASE 2: COORDINATE REGRESSOR [{backbone_name}]")
    print("="*50)
    
    os.makedirs("saved_models", exist_ok=True)
    save_path = f"saved_models/layer2_{backbone_name}.pth"
    
    # MODULAR LAYER 2 RESUME
    if os.path.exists(save_path):
        print(f"--> Found existing Layer 2 weights! Loading {save_path} to resume.")
        layer2_weights = torch.load(save_path, map_location=device, weights_only=True)
        model.coordinate_head.load_state_dict(layer2_weights, strict=False)
    else:
        print("--> No existing Layer 2 weights found. Starting fresh.")
    
    for param in model.backbone.parameters(): param.requires_grad = False
    for param in model.country_head.parameters(): param.requires_grad = False
    for param in model.coordinate_head.parameters(): param.requires_grad = True
        
    persist = True if workers > 0 else False
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=workers, persistent_workers=persist)
    
    criterion = CoordinateLoss()
    optimizer = optim.AdamW(model.coordinate_head.parameters(), lr=lr)
    scaler = torch.amp.GradScaler('cuda')
    num_countries = model.country_head.out_features
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        progress_bar = tqdm(dataloader, desc=f"Phase 2 - Epoch {epoch+1}/{epochs} [{backbone_name}]")
        
        for batch in progress_bar:
            images = batch['image'].to(device, non_blocking=True)
            labels = batch['country_label'].to(device, non_blocking=True)
            lats = batch['latitude'].to(device, non_blocking=True)
            lons = batch['longitude'].to(device, non_blocking=True)
            
            # Target Smoothing
            uniform_prob = 0.002 / (num_countries - 1)
            force_probs = torch.full((images.size(0), num_countries), uniform_prob, device=device)
            force_probs.scatter_(1, labels.unsqueeze(1), 1-uniform_prob*(num_countries-1))
            
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                outputs = model(images, force_country_probs=force_probs)
                loss = criterion(outputs['pred_xyz'], lats, lons)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item()
            progress_bar.set_postfix({'xyz_mse_loss': f"{loss.item():.5f}"})
            
        epoch_loss = running_loss / len(dataloader)
        print(f"--> Epoch {epoch+1} Average Training Loss: {epoch_loss:.5f}")
        torch.save(model.coordinate_head.state_dict(), save_path)
        
    print(f"SUCCESS! Phase 2 Model safely saved to: {save_path}")
    return model

In [10]:
# 6. EVAL FOR LAYER 1 + LAYER 2
def evaluate_phase2_coordinates(model, dataset, batch_size=64, workers=8):
    print("\n--- Evaluating Layer 2 (Coordinate Regressor) ---")
    model.eval()
    eval_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=workers)
    all_distances_km = []
    
    with torch.no_grad():
        for batch in tqdm(eval_loader, desc="Evaluating Distances"):
            images = batch['image'].to(device, non_blocking=True)
            true_lats = batch['latitude'].to(device, non_blocking=True)
            true_lons = batch['longitude'].to(device, non_blocking=True)
            
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                distances = haversine_distance(outputs['pred_lat'], outputs['pred_lon'], true_lats, true_lons)
            all_distances_km.extend(distances.cpu().numpy().tolist())
            
    print(f"--> Median Distance Error: {np.median(all_distances_km):.2f} km (Leaderboard Metric)\n")

In [11]:
# 7 RUN YOUR EXPERIMENTS HERE (LAYER 2)

# Ensure Layer 1 weights are loaded BEFORE starting Layer 2 training!
phase1_weights = torch.load("saved_models/layer1_efficientnetv2_rw_s.pth", map_location=device, weights_only=True)
model_v2_small.backbone.load_state_dict(phase1_weights['backbone'], strict=False)
model_v2_small.country_head.load_state_dict(phase1_weights['country_head'], strict=False)

# Now train the Coordinate Regressor on top of the frozen Layer 1
model_v2_small = train_phase2_coordinates(
    model=model_v2_small, dataset=dataset, backbone_name='efficientnetv2_rw_s',
    epochs=6, batch_size=40, lr=1e-4
)


STARTING PHASE 2: COORDINATE REGRESSOR [efficientnetv2_rw_s]
--> No existing Layer 2 weights found. Starting fresh.


Phase 2 - Epoch 1/6 [efficientnetv2_rw_s]: 100%|██████████| 476/476 [01:05<00:00,  7.26it/s, xyz_mse_loss=0.03641]


--> Epoch 1 Average Training Loss: 0.08439


Phase 2 - Epoch 2/6 [efficientnetv2_rw_s]: 100%|██████████| 476/476 [00:36<00:00, 13.03it/s, xyz_mse_loss=0.46032]


--> Epoch 2 Average Training Loss: 0.03645


...

--> Epoch 4 Average Training Loss: 0.01669


Phase 2 - Epoch 5/6 [efficientnetv2_rw_s]: 100%|██████████| 476/476 [00:37<00:00, 12.75it/s, xyz_mse_loss=0.01373]


--> Epoch 5 Average Training Loss: 0.01373


Phase 2 - Epoch 6/6 [efficientnetv2_rw_s]: 100%|██████████| 476/476 [00:37<00:00, 12.69it/s, xyz_mse_loss=0.50696]


--> Epoch 6 Average Training Loss: 0.01085
SUCCESS! Phase 2 Model safely saved to: saved_models/layer2_efficientnetv2_rw_s.pth


In [12]:
# Evaluate Phase 2
evaluate_phase2_coordinates(model_v2_small, dataset, batch_size=40)


--- Evaluating Layer 2 (Coordinate Regressor) ---


Evaluating Distances: 100%|██████████| 476/476 [01:03<00:00,  7.45it/s]

--> Median Distance Error: 944.82 km (Leaderboard Metric)



In [13]:
# 8. RELOAD & PREPARE INFERENCE
importlib.reload(scripts.model)
from scripts.model import GeoguessrModel
dataset = GeoguessrDataset(
    csv_path='training_dataset/noised_dataset/clustered_training_data.csv',
    image_dir='training_dataset/noised_dataset/images'
)
num_countries = dataset.get_num_classes()
# Empty Chassis
model_v2_small = GeoguessrModel(num_countries=num_countries, backbone_name='efficientnetv2_rw_s', pretrained=False).to(device)

In [ ]:
# 8. BATCH INFERENCE FOR FINAL SUBMISSION WITH CENTROID LOOKUP
class InferenceDataset(Dataset):
    def __init__(self, image_ids, test_image_dir, transform):
        self.image_ids = image_ids
        self.test_image_dir = test_image_dir
        self.transform = transform
    def __len__(self): return len(self.image_ids)
    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img_path = os.path.join(self.test_image_dir, image_id)
        try:
            image_tensor = self.transform(Image.open(img_path).convert('RGB'))
            return image_tensor, image_id, True
        except:
            return torch.zeros((3, 224, 224)), image_id, False
def generate_final_submission(model, dataset, backbone_name, sample_sub_path, test_image_dir, centroids_csv_path,
                              output_path=None, batch_size=64):
    print("\n" + "="*50)
    print("STARTING FAST BATCH INFERENCE WITH CENTROID LOOKUP")
    print("="*50)
    
    if output_path is None:
        output_path = f"submissions/final_submission_{backbone_name}.csv"
        
    device = next(model.parameters()).device
    model.eval()
    
    # 1. Load the cluster centroids for Radius Calibration
    print(f"Loading cluster centroids from {centroids_csv_path}...")
    centroids_df = pd.read_csv(centroids_csv_path)
    centroid_xyz = torch.tensor(centroids_df[['x', 'y', 'z']].values, dtype=torch.float32, device=device)
    centroid_cluster_indices = torch.tensor([dataset.label_mapping[cid] for cid in centroids_df['cluster_id']], device=device)
    
    df = pd.read_csv(sample_sub_path)
    transform = create_transform(**timm.data.resolve_data_config({}, model=timm.create_model(backbone_name, pretrained=False)),
                                 is_training=False)
    
    infer_loader = DataLoader(
        InferenceDataset(df['image_id'].tolist(), test_image_dir, transform), 
        batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True
    )
    
    final_results = []
    
    with torch.no_grad():
        for images, image_ids, valids in tqdm(infer_loader, desc="Predicting"):
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                pred_lats = outputs['pred_lat']
                pred_lons = outputs['pred_lon']
                probs = F.softmax(outputs['country_logits'], dim=1)
                
                # GPU Tensor Math to find the nearest cluster centroid!
                pred_xyz = latlon_to_cartesian(pred_lats, pred_lons)
                dist_sq = torch.sum((pred_xyz.unsqueeze(1) - centroid_xyz.unsqueeze(0)) ** 2, dim=2)
                closest_centroid_idxs = torch.argmin(dist_sq, dim=1)
                
                target_classes = centroid_cluster_indices[closest_centroid_idxs]
                safe_probs = probs[torch.arange(images.size(0), device=device), target_classes].cpu().numpy()
                
            pred_lats_np = pred_lats.cpu().numpy()
            pred_lons_np = pred_lons.cpu().numpy()
            
            for i in range(len(image_ids)):
                img_id = image_ids[i]
                if valids[i]:
                    # Using the SAFE probability of the geographically nearest cluster
                    radius = max(100.0, min(6400.0, 6400 * np.exp(-4.158 * safe_probs[i])))
                    final_results.append({'image_id': img_id, 'pred_lat': pred_lats_np[i], 'pred_lon': pred_lons_np[i],
                                          'pred_radius_km': radius})
                else:
                    final_results.append({'image_id': img_id, 'pred_lat': 0.0, 'pred_lon': 0.0, 'pred_radius_km': 6400.0})
    final_df = pd.DataFrame(final_results).set_index('image_id').reindex(df['image_id']).reset_index()
    final_df.to_csv(output_path, index=False)
    print(f"SUCCESS! Saved to: {os.path.abspath(output_path)}")

In [15]:
# 9. EXECUTE FULL INFERENCE
print("Loading modular weights...")
phase1_weights = torch.load("saved_models/layer1_efficientnetv2_rw_s.pth", map_location=device, weights_only=True)
model_v2_small.backbone.load_state_dict(phase1_weights['backbone'], strict=False)
model_v2_small.country_head.load_state_dict(phase1_weights['country_head'], strict=False)
phase2_weights = torch.load("saved_models/layer2_efficientnetv2_rw_s.pth", map_location=device, weights_only=True)
model_v2_small.coordinate_head.load_state_dict(phase2_weights, strict=False)
print("Successfully loaded all modules into memory!")

generate_final_submission(
    model=model_v2_small, 
    dataset=dataset, 
    backbone_name='efficientnetv2_rw_s', 
    sample_sub_path='sample_submission.csv', 
    test_image_dir='test_images_sampled', 
    centroids_csv_path='training_dataset/noised_dataset/cluster_centroids.csv',
    batch_size=50
)

Loading modular weights...
Successfully loaded all modules into memory!

STARTING FAST BATCH INFERENCE WITH CENTROID LOOKUP
Loading cluster centroids from training_dataset/noised_dataset/cluster_centroids.csv...


Predicting: 100%|██████████| 10/10 [00:30<00:00,  3.09s/it]

SUCCESS! Saved to: c:\Users\Yash T\Desktop\geoguessr\geolocation-prediction\final_submission_efficientnetv2_rw_s.csv
